In [ ]:

# Imports
import pandas as pd
import pyspark.pandas as ps
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import mlflow
#import mlflow.sklearn

# Load original table into pandas
df = spark.read.format("delta").table("gold.data_sample").toPandas()
features = df.columns.drop(["CODE", "total_persons"])

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Elbow method to determine best k
inertia = []
K_range = range(2, 10)
for k in K_range:
    kmeans_tmp = KMeans(n_clusters=k, random_state=42)
    kmeans_tmp.fit(X_scaled)
    inertia.append(kmeans_tmp.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for KMeans Clustering")
plt.show() 

# Fit KMeans with chosen k (4)
kmeans = KMeans(n_clusters=4, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Save KMeans model to MLflow
with mlflow.start_run():
    mlflow.sklearn.log_model(kmeans, "kmeans_model", input_example=X_scaled[:5])
    mlflow.log_param("n_clusters", 4)

print("KMeans model saved to MLflow!")

# Save new base table with cluster column
spark_df = ps.from_pandas(df).to_spark()
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.data_segmented")

print("Delta table gold.data_segmented created with cluster column!")

# Compute and save cluster profile table
spark.createDataFrame(df.groupby("cluster")[features].mean().reset_index()) \
     .write.format("delta") \
     .mode("overwrite") \
     .option("overwriteSchema", "true") \
     .saveAsTable("gold.segments")

print("Cluster profile saved as gold.segments")



,cluster,ave_household_size,med_age,med_hhd_inc_weekly,med_mortgage_monthly,med_prsn_inc_weekly,med_rent_weekly
0,0,2.494017,38.068376,2000.102564,2386.495726,922.512821,472.247863
1,1,2.916000,35.993333,1727.626667,2019.613333,712.220000,399.580000
2,2,2.348039,46.053922,1309.828431,1545.387255,674.583333,307.661765
3,3,2.896429,38.794643,2620.678571,2895.294643,1070.410714,566.303571
